In [6]:
import os
import pickle
import tkinter as tk
from tkinter import ttk, messagebox, scrolledtext
import pandas as pd

# === Load data ===
with open("data/articles_with_all_stemmers.pkl", "rb") as f:
    data = pickle.load(f)

ngram_base_path = "data/N-Gram_results"

# === Helper functions ===
def list_volumes():
    return list(data.keys())

def list_issues(volume):
    return list(data[volume].keys())

def list_articles(volume, issue):
    return range(1, len(data[volume][issue]) + 1)

def load_ngram(volume, issue, article_num, ngram_type):
    folder_name = f"{volume.replace(' ', '_')}_{issue.replace(' ', '_')}_Article{article_num}"
    file_path = os.path.join(ngram_base_path, folder_name, f"{ngram_type}.pkl")
    if not os.path.exists(file_path):
        return None
    with open(file_path, "rb") as f:
        return pickle.load(f)

# === GUI ===
root = tk.Tk()
root.title("Article Explorer")
root.geometry("800x600")

# --- Dropdowns ---
volume_var = tk.StringVar()
issue_var = tk.StringVar()
article_var = tk.StringVar()
stemmer_var = tk.StringVar(value="snowball")
ngram_var = tk.StringVar(value="unigram")

# --- Frame setup ---
frame_top = ttk.Frame(root, padding=10)
frame_top.pack(fill="x")

frame_output = ttk.Frame(root, padding=10)
frame_output.pack(fill="both", expand=True)

output_box = scrolledtext.ScrolledText(frame_output, wrap=tk.WORD, height=25)
output_box.pack(fill="both", expand=True)

# --- Dropdowns ---
ttk.Label(frame_top, text="Volume:").grid(row=0, column=0)
volume_combo = ttk.Combobox(frame_top, textvariable=volume_var, values=list_volumes(), width=25)
volume_combo.grid(row=0, column=1)

ttk.Label(frame_top, text="Issue:").grid(row=0, column=2)
issue_combo = ttk.Combobox(frame_top, textvariable=issue_var, width=25)
issue_combo.grid(row=0, column=3)

ttk.Label(frame_top, text="Article #:").grid(row=0, column=4)
article_combo = ttk.Combobox(frame_top, textvariable=article_var, width=5)
article_combo.grid(row=0, column=5)

# --- Update issue list when volume changes ---
def update_issues(event):
    vol = volume_var.get()
    if vol:
        issue_combo["values"] = list_issues(vol)
volume_combo.bind("<<ComboboxSelected>>", update_issues)

# --- Update article list when issue changes ---
def update_articles(event):
    vol = volume_var.get()
    iss = issue_var.get()
    if vol and iss:
        article_combo["values"] = list_articles(vol, iss)
issue_combo.bind("<<ComboboxSelected>>", update_articles)

# --- Display functions ---
def show_article():
    vol, iss, art = volume_var.get(), issue_var.get(), article_var.get()
    if not (vol and iss and art):
        messagebox.showerror("Error", "Please select volume, issue, and article.")
        return
    art = int(art)
    article = data[vol][iss][art - 1]
    output_box.delete(1.0, tk.END)
    output_box.insert(tk.END, f"📄 TITLE:\n{article['title']}\n\n🧾 ABSTRACT:\n{article['abstract']}")

def show_tokens():
    vol, iss, art = volume_var.get(), issue_var.get(), article_var.get()
    stem = stemmer_var.get().lower()
    if not (vol and iss and art):
        messagebox.showerror("Error", "Please select volume, issue, and article.")
        return
    art = int(art)
    article = data[vol][iss][art - 1]
    key = f"tokens_{stem}"
    tokens = article.get(key, [])
    output_box.delete(1.0, tk.END)
    output_box.insert(tk.END, f"🔤 Tokens ({stem} stemmer):\n\n")
    output_box.insert(tk.END, ", ".join(tokens[:100]) + "...\n\n")
    output_box.insert(tk.END, f"Total tokens: {len(tokens)}")

def show_ngram():
    vol, iss, art = volume_var.get(), issue_var.get(), article_var.get()
    ngram = ngram_var.get().lower()
    if not (vol and iss and art):
        messagebox.showerror("Error", "Please select volume, issue, and article.")
        return
    art = int(art)
    ngram_data = load_ngram(vol, iss, art, ngram)
    output_box.delete(1.0, tk.END)
    if ngram_data is None:
        output_box.insert(tk.END, "⚠️ No N-gram data found for this article.")
        return
    df = pd.DataFrame([(str(k), v[0], v[1]) for k, v in ngram_data.items()],
                      columns=["Token(s)", "Frequency", "Probability"])
    output_box.insert(tk.END, f"📊 {ngram.capitalize()} Model Results:\n\n")
    output_box.insert(tk.END, df.head(20).to_string(index=False))
    output_box.insert(tk.END, f"\n\nTotal unique {ngram}s: {len(df)}")

# --- Control buttons ---
ttk.Button(frame_top, text="Show Article", command=show_article).grid(row=1, column=0, pady=10)
ttk.Label(frame_top, text="Stemmer:").grid(row=1, column=1)
ttk.Combobox(frame_top, textvariable=stemmer_var, values=["snowball", "porter", "lancaster"], width=10).grid(row=1, column=2)
ttk.Button(frame_top, text="Show Tokens", command=show_tokens).grid(row=1, column=3)

ttk.Label(frame_top, text="N-gram:").grid(row=1, column=4)
ttk.Combobox(frame_top, textvariable=ngram_var, values=["unigram", "bigram", "trigram"], width=10).grid(row=1, column=5)
ttk.Button(frame_top, text="Show N-gram", command=show_ngram).grid(row=1, column=6)

root.mainloop()
